# Financial Fraud Detection

## Phase 13 - Real-Time Fraud Prediction

This notebook:

- Connects Spark Streaming to Kafka
- Reads live transactions
- Parses JSON
- Loads the trained Random Forest model
- Predicts fraud in real time

## Imports

In [33]:
import json
import joblib
import pandas as pd

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    from_json,
    when,
    log1p
)

from pyspark.sql.types import *

## Layout Cleaning

In [34]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

## Create Spark Session

In [35]:
spark = (
    SparkSession.builder
    .appName("RealTimeFraudPrediction")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1"
    )
    .getOrCreate()
)

## Stop ALL running queries

In [36]:
for q in spark.streams.active:
    print("Stopping:", q.id)
    q.stop()

ERROR:py4j.clientserver:There was an exception while executing the Python Proxy on the Python Side.
Traceback (most recent call last):
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/py4j/clientserver.py", line 644, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/pyspark/sql/utils.py", line 165, in call
    raise e
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/pyspark/sql/utils.py", line 162, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/2s/z01cdgj13ydbbdpl84qrhzsr0000gn/T/ipykernel_50773/2519437040.py", line 3, in predict_batch
    pdf = batch_df.toPandas()
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Frau

Stopping: baf6e26d-7c35-42ae-99d2-82b51b5f3294


Exception in thread "stream execution thread for [id = baf6e26d-7c35-42ae-99d2-82b51b5f3294, runId = 063cb142-3231-41c3-b8d2-309d378d8740]" java.lang.StackOverflowError
	at java.base/java.util.regex.IntHashSet.contains(IntHashSet.java:47)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:4921)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:4847)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4725)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:3958)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4761)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4816)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:4925)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:4847)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4725)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:3958)
	at java.base/java.util.regex.Patter

In [37]:
spark.streams.active

[]

## Load Best Model

In [38]:
model = joblib.load("../models/best_model.pkl")

print(model)

RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)


## Load Metadata

In [39]:
with open("../models/best_model_metadata.json","r") as f:
    metadata = json.load(f)

metadata

{'model_name': 'Random Forest (Class Weights)',
 'filename': 'best_model.pkl',
 'algorithm': 'RandomForestClassifier',
 'accuracy': 0.999489,
 'precision': 0.958333,
 'recall': 0.726316,
 'f1_score': 0.826347,
 'roc_auc': 0.943893,
 'random_state': 42,
 'scikit_learn_version': '1.9.0',
 'saved_on': '2026-07-11 20:13:28'}

## Read Kafka Stream

In [40]:
stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers","localhost:9092")
    .option("subscribe","fraud-transactions")
    .load()
)

## Schema

In [41]:
schema = StructType([

    StructField("Time", DoubleType()),

    StructField("V1", DoubleType()),
    StructField("V2", DoubleType()),
    StructField("V3", DoubleType()),
    StructField("V4", DoubleType()),
    StructField("V5", DoubleType()),
    StructField("V6", DoubleType()),
    StructField("V7", DoubleType()),
    StructField("V8", DoubleType()),
    StructField("V9", DoubleType()),
    StructField("V10", DoubleType()),
    StructField("V11", DoubleType()),
    StructField("V12", DoubleType()),
    StructField("V13", DoubleType()),
    StructField("V14", DoubleType()),
    StructField("V15", DoubleType()),
    StructField("V16", DoubleType()),
    StructField("V17", DoubleType()),
    StructField("V18", DoubleType()),
    StructField("V19", DoubleType()),
    StructField("V20", DoubleType()),
    StructField("V21", DoubleType()),
    StructField("V22", DoubleType()),
    StructField("V23", DoubleType()),
    StructField("V24", DoubleType()),
    StructField("V25", DoubleType()),
    StructField("V26", DoubleType()),
    StructField("V27", DoubleType()),
    StructField("V28", DoubleType()),

    StructField("Amount", DoubleType()),
    StructField("transaction_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("Class", DoubleType())
])

## Parse JSON

In [42]:
parsed_df = (
    stream_df
    .selectExpr("CAST(value AS STRING)")
    .select(
        from_json(
            col("value"),
            schema
        ).alias("data")
    )
    .select("data.*")
)

## Verify Schema

In [43]:
parsed_df.printSchema()

root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

## Feature Engineering

In [44]:
feature_df = (

    parsed_df

    .withColumn(
        "Large_Transaction",
        when(col("Amount") > 200, 1).otherwise(0)
    )

    .withColumn(
        "Log_Amount",
        log1p(col("Amount"))
    )

    .withColumn(
        "Amount_Level",
        when(col("Amount") < 10, 0)
        .when(col("Amount") < 100, 1)
        .when(col("Amount") < 500, 2)
        .otherwise(3)
    )

)

26/07/12 12:59:10 WARN TaskSetManager: Lost task 0.0 in stage 98.0 (TID 98) (192.168.29.167 executor driver): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED] Job 98 cancelled Query [id = baf6e26d-7c35-42ae-99d2-82b51b5f3294, runId = 063cb142-3231-41c3-b8d2-309d378d8740] was stopped SQLSTATE: XXKDA)


## Select Features in Model Order

In [45]:
feature_columns = list(model.feature_names_in_)

prediction_df = feature_df.select(

    "transaction_id",

    "event_timestamp",

    *feature_columns

)

## Verify Columns

In [46]:
prediction_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)


## Create Prediction Function

In [47]:
def predict_batch(batch_df, batch_id):

    pdf = batch_df.toPandas()

    if pdf.empty:
        return

    # Preserve metadata
    metadata_df = pdf[
        [
            "transaction_id",
            "event_timestamp"
        ]
    ].copy()

    # Features for prediction
    X = pdf[model.feature_names_in_]

    # Predictions
    predictions = model.predict(X)

    probabilities = model.predict_proba(X)[:, 1]

    # Final output
    result = metadata_df.copy()

    result["Amount"] = pdf["Amount"]

    result["Prediction"] = predictions

    result["Status"] = result["Prediction"].map({
        0: "✅ Genuine",
        1: "🚨 Fraud"
    })

    result["Fraud Probability"] = probabilities.round(6)

    result["Risk Level"] = pd.cut(
        result["Fraud Probability"],
        bins=[-0.01, 0.20, 0.50, 0.80, 1.00],
        labels=[
            "🟢 Low",
            "🟡 Medium",
            "🟠 High",
            "🔴 Critical"
        ]
    )

    print("\n")
    print("=" * 120)
    print(f"Batch {batch_id}")
    print("=" * 120)

    print(
        result[
            [
                "transaction_id",
                "Amount",
                "Fraud Probability",
                "Risk Level",
                "Status"
            ]
        ]
    )

    frauds = result[result["Prediction"] == 1]

    if not frauds.empty:

        print("\n")
        print("🚨" * 25)
        print("🚨 LIVE FRAUD ALERT 🚨")
        print("🚨" * 25)

        print(
            frauds[
                [
                    "transaction_id",
                    "Amount",
                    "Fraud Probability",
                    "Risk Level"
                ]
            ]
        )

    print("\n")
    print(f"Processed {len(result)} transaction(s)")

## Start Streaming Prediction

In [48]:
query = (

    prediction_df.writeStream

    .foreachBatch(predict_batch)

    .outputMode("append")

    .option(
        "checkpointLocation",
        "../artifacts/checkpoints/realtime_prediction"
    )

    .start()

)

26/07/12 12:59:10 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## Streaming Status

In [49]:
print(query.status)

{'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


## Active Streams

In [50]:
spark.streams.active

## Keep Alive

In [51]:
query.awaitTermination()



Batch 728
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  70ec6127-2ac7-4c93-9933-75664a89c77f  378.66                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 729
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  002efcf3-9e97-4c8c-9539-7d85af0725f0   123.5                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 730
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  16fda74b-84a0-466c-9527-a5c110c7b5de   69.99                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 731
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  7c426ad7-5e01-48f8-a2b7-55b899c06c3f    3.67                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 732
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  2871198f-6fbd-4462-8



Batch 763
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  e1cbcdab-7d65-45c5-b5ba-60f7b5d539f6     1.8                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 764
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  b8135218-7d0c-4463-bc2e-0742f3bdc91a   20.53                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)




Batch 765
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  3b5b64dd-461c-4c39-9560-d262c0637511    6.54                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 766
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  f61cf50b-abb6-4a59-af63-76aa163c30bc   29.89                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 767
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  25efb976-ad26-4a53-a232-7bd52ebef81f    2.35                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 768
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  2df9d407-8e55-49cd-9e8d-ca000f1bec6c    14.8                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 769
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  44c01de4-1594-43fb-8



Batch 805
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  f6f0d035-058e-4a1e-9506-edee08558c53    14.8                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 806
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  a0fea82e-aeca-4c4a-88ae-a4e5ccc97888    1.98                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 807
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  c6afd494-461b-44c4-a395-5c3319b4188d    6.67                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 808
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  239cb277-f54d-4f8b-97d0-ddbd10d7f872    1.46                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)




Batch 809
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  adfea762-d680-4354-8fcb-e04e23d94e04   89.17                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 810
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  d79c0088-39e5-460e-a6fa-80e61637bd99    7.53                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 811
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  8a7330fa-fb3e-4b22-8043-0e9076e7f98f  200.01               0.01      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 812
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  09936c3d-a4a4-4e93-8849-939f1d584c11    0.76               0.02      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 813
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  42dbf47b-a42d-44c0-9

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/anaconda3/lib/python3.13/socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 

## Run to End the stream

In [ ]:
query.stop()

print("Streaming stopped.")

In [ ]:
spark.streams.active